# Semantic Similarity Filtering

Notebook này dùng để lọc các mẫu hỏi - đáp quá giống nhau sau bước tăng cường dữ liệu. Vì dữ liệu sinh tự động dễ bị lặp cấu trúc câu, bước lọc tương đồng giúp giảm trùng lặp và giữ lại các mẫu có giá trị hơn cho huấn luyện.

Phương pháp sử dụng mô hình embedding đa ngôn ngữ để biểu diễn câu hỏi và câu trả lời thành vector, sau đó tính độ tương đồng ngữ nghĩa. Việc so sánh ưu tiên thực hiện trong cùng nhóm ảnh, nhãn bệnh và loại câu hỏi để tránh loại nhầm các mẫu giống câu chữ nhưng khác ngữ cảnh.

In [ ]:
from google.colab import drive
# Optional when running on Colab:
# from google.colab import drive
# drive.mount('./data_sample')

In [ ]:
import os
import re
import sys
import json
import hashlib
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Cài sentence-transformers nếu chưa có
try:
    from sentence_transformers import SentenceTransformer
except Exception:
    print("Đang cài sentence-transformers...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

from sklearn.neighbors import NearestNeighbors

# CẤU HÌNH ĐƯỜNG DẪN
INPUT_DIR = "./outputs/spcqa_augmented"

# Thư mục lưu file sau lọc similarity.
OUTPUT_DIR = "./outputs/spcqa_similarity_safe_filter_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMBED_CACHE_DIR = os.path.join(OUTPUT_DIR, "embedding_cache")
os.makedirs(EMBED_CACHE_DIR, exist_ok=True)

FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

FILE_NAMES = {
    "ca_chua": "ca_chua_train_qa_spcqa_v4_reference_32000.csv",
    "bi_do": "bi_do_train_qa_spcqa_v4_best_32000.csv",
    "kho_qua": "kho_qua_train_qa_spcqa_v4_best_32000.csv",
    "dua_leo": "dua_leo_train_qa_spcqa_v4_best_32000.csv",
}

# CẤU HÌNH MODEL EMBEDDING

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# CẤU HÌNH LỌC AN TOÀN

# Chỉ dùng question để tránh target_answer template làm similarity tăng quá cao.
# Có thể đổi thành "question_answer" nếu muốn đánh giá cả câu trả lời.
TEXT_MODE = "question_answer"

# Chỉ lọc near-duplicate trong cùng ảnh + cùng nhãn + cùng question_type.
# Không so câu của ảnh khác nhau, vì sẽ dễ xóa quá nhiều.
NEAR_DUP_GROUP_COLS = ["image_id", "canonical_label", "question_type"]

# Outlier review theo nhóm rộng hơn.
OUTLIER_GROUP_COLS = ["canonical_label", "question_type"]

# Ngưỡng near-duplicate rất cao để chỉ xóa câu gần như giống nhau.
TOO_SIMILAR_THRESHOLD = 0.9

# Outlier dùng percentile thấp nhất trong từng nhóm.
# Mặc định chỉ gắn cờ review, không xóa tự động.
TOO_DIFFERENT_PERCENTILE = 1.0
AUTO_REMOVE_OUTLIER = True

# Chỉ xóa dòng augment, giữ dòng gốc/reference.
REMOVE_ONLY_AUGMENT = True

# Không lọc cà chua reference vì đây là mốc chuẩn 32.000.
SKIP_FILTER_CROPS = ["ca_chua"]

# Chặn xóa quá nhiều. Nếu số dòng bị đề xuất xóa vượt quá tỷ lệ này,
# code chỉ xóa những dòng có nearest_similarity cao nhất.
MAX_REMOVE_RATIO = 0.03

print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_NAME:", MODEL_NAME)
print("TEXT_MODE:", TEXT_MODE)
print("TOO_SIMILAR_THRESHOLD:", TOO_SIMILAR_THRESHOLD)
print("AUTO_REMOVE_OUTLIER:", AUTO_REMOVE_OUTLIER)
print("MAX_REMOVE_RATIO:", MAX_REMOVE_RATIO)


In [ ]:
def read_input_files(input_dir, file_names):
    dfs = {}

    for crop_key, file_name in file_names.items():
        path = os.path.join(input_dir, file_name)

        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Không tìm thấy file: {path}\n"
                f"Hãy kiểm tra lại INPUT_DIR hoặc tên file trong FILE_NAMES."
            )

        df = pd.read_csv(path)
        dfs[crop_key] = df
        print(f"{crop_key}: {df.shape}")

    return dfs


dfs = read_input_files(INPUT_DIR, FILE_NAMES)

BASE_COLS = dfs["ca_chua"].columns.tolist()
CONTENT_COLS = [c for c in BASE_COLS if c != "sample_id"]

print("\nSố cột chuẩn:", len(BASE_COLS))
print(BASE_COLS)

# Kiểm tra schema ban đầu
schema_rows = []
for crop_key, df in dfs.items():
    schema_rows.append({
        "crop": crop_key,
        "rows": len(df),
        "columns": len(df.columns),
        "schema_ok": list(df.columns) == BASE_COLS,
        "duplicate_columns": int(df.columns.duplicated().sum()),
        "duplicate_strict": int(df.duplicated(subset=CONTENT_COLS).sum()),
        "duplicate_image_question_answer": int(df.duplicated(subset=["image_id", "question", "target_answer"]).sum()),
        "unique_questions": int(df["question"].nunique()),
        "unique_question_types": int(df["question_type"].nunique()),
    })

schema_df = pd.DataFrame(schema_rows)
print("\nKiểm tra trước lọc:")
print(schema_df)

assert all(schema_df["schema_ok"]), "Có file bị lệch schema trước khi lọc."
assert schema_df["duplicate_columns"].sum() == 0, "Có file bị duplicate cột."


In [ ]:
model = SentenceTransformer(MODEL_NAME)
print("Loaded model:", MODEL_NAME)


In [ ]:
def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def normalize_space(text):
    text = safe_text(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def is_augmented_sample(sample_id):
    sample_id = safe_text(sample_id).upper()
    return "SPCQA" in sample_id or "AUG" in sample_id or "V4" in sample_id


def make_semantic_text(row, mode="question_only"):
    question = normalize_space(row["question"])
    answer = normalize_space(row["target_answer"])

    if mode == "question_answer":
        return f"Câu hỏi: {question} Câu trả lời: {answer}"

    return question


def make_text_hash(texts, model_name, text_mode):
    h = hashlib.sha256()
    h.update(model_name.encode("utf-8"))
    h.update(b"\0")
    h.update(text_mode.encode("utf-8"))
    h.update(b"\0")
    h.update(str(len(texts)).encode("utf-8"))
    h.update(b"\0")

    for text in texts:
        h.update(text.encode("utf-8", errors="ignore"))
        h.update(b"\0")

    return h.hexdigest()


def encode_texts_with_cache(crop_key, texts, model, cache_dir, model_name, text_mode):
    digest = make_text_hash(texts, model_name=model_name, text_mode=text_mode)
    cache_path = os.path.join(cache_dir, f"{crop_key}_{text_mode}_{digest[:16]}.npy")

    if os.path.exists(cache_path):
        embeddings = np.load(cache_path)
        if embeddings.shape[0] == len(texts):
            print(f"Load cached embeddings: {crop_key} {embeddings.shape}")
            return embeddings

    print(f"Encoding embeddings: {crop_key} - {len(texts)} texts")

    embeddings = model.encode(
        texts,
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    embeddings = np.asarray(embeddings, dtype=np.float32)
    np.save(cache_path, embeddings)

    return embeddings


def compute_safe_near_duplicate_flags(df, embeddings, group_cols, threshold):
    """
    Near-duplicate filter an toàn:
    - Chỉ so trong cùng image_id + canonical_label + question_type.
    - Ưu tiên giữ dòng gốc trước.
    - Chỉ đánh dấu dòng augment nếu quá giống dòng đã giữ.
    """
    n = len(df)

    too_similar = np.zeros(n, dtype=bool)
    nearest_similarity = np.zeros(n, dtype=np.float32)
    nearest_index = np.full(n, -1, dtype=int)

    work_df = df.reset_index(drop=True).copy()
    work_df["_row_index"] = np.arange(n)
    work_df["_is_augmented"] = work_df["sample_id"].apply(is_augmented_sample)

    for _, group_df in work_df.groupby(group_cols, dropna=False):
        idxs = group_df["_row_index"].values

        if len(idxs) < 2:
            continue

        group_emb = embeddings[idxs]

        nn = NearestNeighbors(
            n_neighbors=2,
            metric="cosine",
            algorithm="brute"
        )
        nn.fit(group_emb)

        distances, indices = nn.kneighbors(group_emb)
        sims = 1 - distances[:, 1]
        local_nearest = indices[:, 1]

        for local_i, sim in enumerate(sims):
            global_i = idxs[local_i]
            global_j = idxs[local_nearest[local_i]]
            nearest_similarity[global_i] = float(sim)
            nearest_index[global_i] = int(global_j)

        # Greedy keep: dòng gốc trước, augment sau
        group_order = group_df.sort_values(
            by=["_is_augmented", "sample_id"],
            ascending=[True, True]
        )["_row_index"].values

        kept_embs = []

        for idx in group_order:
            is_aug = bool(work_df.loc[idx, "_is_augmented"])

            if len(kept_embs) == 0:
                kept_embs.append(embeddings[idx])
                continue

            sims_to_kept = np.dot(np.vstack(kept_embs), embeddings[idx])
            max_sim = float(np.max(sims_to_kept))

            if is_aug and max_sim >= threshold:
                too_similar[idx] = True
            else:
                kept_embs.append(embeddings[idx])

    return too_similar, nearest_similarity, nearest_index


def compute_soft_outlier_flags(df, embeddings, group_cols, percentile=1.0):
    """
    Outlier mềm:
    - Tính similarity với centroid trong nhóm canonical_label + question_type.
    - Gắn cờ các dòng thuộc percentile thấp nhất.
    - Mặc định chỉ audit, không tự động xóa.
    """
    n = len(df)

    centroid_similarity = np.ones(n, dtype=np.float32)
    outlier_flag = np.zeros(n, dtype=bool)

    work_df = df.reset_index(drop=True).copy()
    work_df["_row_index"] = np.arange(n)

    for _, group_df in work_df.groupby(group_cols, dropna=False):
        idxs = group_df["_row_index"].values

        if len(idxs) < 30:
            continue

        group_emb = embeddings[idxs]
        centroid = group_emb.mean(axis=0)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-12)

        sims = np.dot(group_emb, centroid)
        cutoff = np.percentile(sims, percentile)

        centroid_similarity[idxs] = sims
        outlier_flag[idxs] = sims <= cutoff

    return outlier_flag, centroid_similarity


def apply_remove_cap(remove_flag, nearest_similarity, max_remove_ratio):
    """
    Chặn xóa quá nhiều dòng. Nếu vượt max_remove_ratio,
    chỉ xóa các dòng có nearest_similarity cao nhất.
    """
    remove_flag = remove_flag.copy()
    n = len(remove_flag)
    max_remove = int(n * max_remove_ratio)

    if remove_flag.sum() <= max_remove:
        return remove_flag

    candidate_idxs = np.where(remove_flag)[0]
    sorted_candidates = sorted(
        candidate_idxs,
        key=lambda i: nearest_similarity[i],
        reverse=True
    )

    keep_remove_idxs = set(sorted_candidates[:max_remove])

    capped_remove = np.zeros(n, dtype=bool)
    for idx in keep_remove_idxs:
        capped_remove[idx] = True

    print(
        f"Cảnh báo: remove_flag ban đầu = {remove_flag.sum()}, "
        f"vượt giới hạn {max_remove}. Đã cap còn {capped_remove.sum()}."
    )

    return capped_remove


In [ ]:
def filter_similarity_safe_one_file(df, crop_key):
    df = df.copy().reset_index(drop=True)

    assert list(df.columns) == BASE_COLS, f"{crop_key}: schema bị lệch."

    df["_is_augmented"] = df["sample_id"].apply(is_augmented_sample)

    texts = [
        make_semantic_text(row, mode=TEXT_MODE)
        for _, row in df.iterrows()
    ]

    embeddings = encode_texts_with_cache(
        crop_key=crop_key,
        texts=texts,
        model=model,
        cache_dir=EMBED_CACHE_DIR,
        model_name=MODEL_NAME,
        text_mode=TEXT_MODE
    )

    too_similar, nearest_similarity, nearest_index = compute_safe_near_duplicate_flags(
        df=df[BASE_COLS],
        embeddings=embeddings,
        group_cols=NEAR_DUP_GROUP_COLS,
        threshold=TOO_SIMILAR_THRESHOLD
    )

    too_different, centroid_similarity = compute_soft_outlier_flags(
        df=df[BASE_COLS],
        embeddings=embeddings,
        group_cols=OUTLIER_GROUP_COLS,
        percentile=TOO_DIFFERENT_PERCENTILE
    )

    if AUTO_REMOVE_OUTLIER:
        remove_flag = too_similar | (too_different & df["_is_augmented"].values)
    else:
        remove_flag = too_similar

    if REMOVE_ONLY_AUGMENT:
        remove_flag = remove_flag & df["_is_augmented"].values

    remove_flag_before_cap = remove_flag.copy()
    remove_flag = apply_remove_cap(
        remove_flag=remove_flag,
        nearest_similarity=nearest_similarity,
        max_remove_ratio=MAX_REMOVE_RATIO
    )

    remove_reason = []
    for i in range(len(df)):
        if remove_flag[i]:
            if too_similar[i]:
                remove_reason.append("remove_too_similar")
            elif too_different[i]:
                remove_reason.append("remove_too_different")
            else:
                remove_reason.append("remove_other")
        else:
            if too_different[i]:
                remove_reason.append("keep_but_outlier_review")
            elif remove_flag_before_cap[i] and not remove_flag[i]:
                remove_reason.append("keep_due_to_remove_cap")
            else:
                remove_reason.append("keep")

    audit_df = df.copy()
    audit_df["semantic_text"] = texts
    audit_df["nearest_similarity"] = nearest_similarity
    audit_df["nearest_row_index"] = nearest_index
    audit_df["centroid_similarity"] = centroid_similarity
    audit_df["too_similar_flag"] = too_similar
    audit_df["too_different_review_flag"] = too_different
    audit_df["remove_flag_before_cap"] = remove_flag_before_cap
    audit_df["remove_flag"] = remove_flag
    audit_df["remove_reason"] = remove_reason

    clean_df = audit_df.loc[~audit_df["remove_flag"], BASE_COLS].copy().reset_index(drop=True)

    summary = {
        "crop": crop_key,
        "rows_before": int(len(df)),
        "rows_after": int(len(clean_df)),
        "removed_total": int(remove_flag.sum()),
        "removed_before_cap": int(remove_flag_before_cap.sum()),
        "removed_too_similar": int((remove_flag & too_similar).sum()),
        "outlier_review_only": int(too_different.sum()),
        "augmented_before": int(df["_is_augmented"].sum()),
        "augmented_after": int(clean_df["sample_id"].apply(is_augmented_sample).sum()),
        "unique_questions_before": int(df["question"].nunique()),
        "unique_questions_after": int(clean_df["question"].nunique()),
        "duplicate_strict_after": int(clean_df.duplicated(subset=CONTENT_COLS).sum()),
        "duplicate_image_question_answer_after": int(clean_df.duplicated(subset=["image_id", "question", "target_answer"]).sum()),
        "unique_question_types_after": int(clean_df["question_type"].nunique()),
    }

    return clean_df, audit_df, summary


In [ ]:
clean_dfs = {}
audit_dfs = {}
summary_rows = []

for crop_key, df in dfs.items():
    print("=" * 100)
    print(f"Đang xử lý similarity an toàn: {crop_key}")

    if crop_key in SKIP_FILTER_CROPS:
        print(f"Bỏ qua lọc {crop_key}, giữ nguyên reference.")

        clean_df = df[BASE_COLS].copy().reset_index(drop=True)

        audit_df = df.copy()
        audit_df["semantic_text"] = audit_df.apply(lambda row: make_semantic_text(row, mode=TEXT_MODE), axis=1)
        audit_df["nearest_similarity"] = np.nan
        audit_df["nearest_row_index"] = -1
        audit_df["centroid_similarity"] = np.nan
        audit_df["too_similar_flag"] = False
        audit_df["too_different_review_flag"] = False
        audit_df["remove_flag_before_cap"] = False
        audit_df["remove_flag"] = False
        audit_df["remove_reason"] = "skip_reference"

        summary = {
            "crop": crop_key,
            "rows_before": int(len(df)),
            "rows_after": int(len(clean_df)),
            "removed_total": 0,
            "removed_before_cap": 0,
            "removed_too_similar": 0,
            "outlier_review_only": 0,
            "augmented_before": int(df["sample_id"].apply(is_augmented_sample).sum()),
            "augmented_after": int(clean_df["sample_id"].apply(is_augmented_sample).sum()),
            "unique_questions_before": int(df["question"].nunique()),
            "unique_questions_after": int(clean_df["question"].nunique()),
            "duplicate_strict_after": int(clean_df.duplicated(subset=CONTENT_COLS).sum()),
            "duplicate_image_question_answer_after": int(clean_df.duplicated(subset=["image_id", "question", "target_answer"]).sum()),
            "unique_question_types_after": int(clean_df["question_type"].nunique()),
        }

    else:
        clean_df, audit_df, summary = filter_similarity_safe_one_file(df, crop_key)

    clean_dfs[crop_key] = clean_df
    audit_dfs[crop_key] = audit_df
    summary_rows.append(summary)

    clean_path = os.path.join(OUTPUT_DIR, f"{crop_key}_spcqa_v4_similarity_safe_clean.csv")
    audit_path = os.path.join(OUTPUT_DIR, f"{crop_key}_spcqa_v4_similarity_safe_audit.csv")

    clean_df.to_csv(clean_path, index=False, encoding="utf-8-sig")
    audit_df.to_csv(audit_path, index=False, encoding="utf-8-sig")

    print("Saved clean:", clean_path)
    print("Saved audit:", audit_path)
    print(summary)

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUTPUT_DIR, "safe_similarity_filter_summary.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\nSummary:")
print(summary_df)
print("Saved summary:", summary_path)


In [ ]:
post_checks = []

for crop_key, df in clean_dfs.items():
    post_checks.append({
        "crop": crop_key,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "schema_ok": list(df.columns) == BASE_COLS,
        "duplicate_columns": int(df.columns.duplicated().sum()),
        "duplicate_strict": int(df.duplicated(subset=CONTENT_COLS).sum()),
        "duplicate_image_question_answer": int(df.duplicated(subset=["image_id", "question", "target_answer"]).sum()),
        "unique_images": int(df["image_id"].nunique()),
        "unique_labels": int(df["canonical_label"].nunique()),
        "unique_question_types": int(df["question_type"].nunique()),
        "unique_questions": int(df["question"].nunique()),
    })

post_check_df = pd.DataFrame(post_checks)
post_check_path = os.path.join(OUTPUT_DIR, "safe_similarity_post_check.csv")
post_check_df.to_csv(post_check_path, index=False, encoding="utf-8-sig")

print("\nPost check:")
print(post_check_df)
print("Saved post check:", post_check_path)

assert all(post_check_df["schema_ok"]), "Có file bị lệch schema sau lọc."
assert post_check_df["duplicate_columns"].sum() == 0, "Có duplicate cột sau lọc."
assert post_check_df["duplicate_strict"].sum() == 0, "Có duplicate strict sau lọc."
assert post_check_df["duplicate_image_question_answer"].sum() == 0, "Có duplicate image+question+answer sau lọc."

print("Kiểm tra sau lọc similarity đạt.")


In [ ]:
def save_current_figure(file_name):
    path = os.path.join(FIGURE_DIR, file_name)
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    print("Saved figure:", path)

# 1. So sánh số dòng trước/sau lọc
ax = summary_df.set_index("crop")[["rows_before", "rows_after"]].plot(kind="bar", figsize=(8, 5))
ax.set_title("So sánh số dòng trước và sau lọc similarity an toàn")
ax.set_xlabel("Loại cây")
ax.set_ylabel("Số dòng")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
save_current_figure("compare_rows_before_after_similarity_safe.png")
plt.show()

# 2. Số dòng bị loại
ax = summary_df.set_index("crop")["removed_total"].plot(kind="bar", figsize=(8, 5))
ax.set_title("Số dòng bị loại bởi lọc similarity an toàn")
ax.set_xlabel("Loại cây")
ax.set_ylabel("Số dòng bị loại")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
save_current_figure("removed_total_similarity_safe.png")
plt.show()

# 3. Outlier chỉ review
ax = summary_df.set_index("crop")["outlier_review_only"].plot(kind="bar", figsize=(8, 5))
ax.set_title("Số dòng được gắn cờ outlier để review")
ax.set_xlabel("Loại cây")
ax.set_ylabel("Số dòng review")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
save_current_figure("outlier_review_only_similarity_safe.png")
plt.show()

# 4. Histogram nearest similarity theo từng cây
for crop_key, audit_df in audit_dfs.items():
    values = audit_df["nearest_similarity"].dropna()
    if len(values) == 0:
        continue

    plt.figure(figsize=(8, 5))
    values.hist(bins=50)
    plt.axvline(TOO_SIMILAR_THRESHOLD, linestyle="--")
    plt.title(f"Phân bố nearest similarity an toàn - {crop_key}")
    plt.xlabel("Cosine similarity")
    plt.ylabel("Số dòng")
    plt.grid(axis="y", alpha=0.3)
    save_current_figure(f"hist_nearest_similarity_{crop_key}.png")
    plt.show()

# 5. Phân bố nhãn sau lọc
for crop_key, clean_df in clean_dfs.items():
    counts = clean_df["canonical_label"].value_counts().sort_values(ascending=True)

    plt.figure(figsize=(10, max(4, len(counts) * 0.5)))
    counts.plot(kind="barh")
    plt.title(f"Phân bố nhãn sau lọc similarity - {crop_key}")
    plt.xlabel("Số dòng")
    plt.ylabel("Nhãn")
    plt.grid(axis="x", alpha=0.3)
    save_current_figure(f"label_distribution_after_filter_{crop_key}.png")
    plt.show()

# 6. Phân bố question_type sau lọc
for crop_key, clean_df in clean_dfs.items():
    counts = clean_df["question_type"].value_counts().sort_values(ascending=True)

    plt.figure(figsize=(10, max(4, len(counts) * 0.5)))
    counts.plot(kind="barh")
    plt.title(f"Phân bố question_type sau lọc similarity - {crop_key}")
    plt.xlabel("Số dòng")
    plt.ylabel("Question type")
    plt.grid(axis="x", alpha=0.3)
    save_current_figure(f"question_type_distribution_after_filter_{crop_key}.png")
    plt.show()


In [ ]:
# GỘP 4 FILE CLEAN THÀNH 1 FILE DUY NHẤT

import os
import pandas as pd

# Thư mục chứa 4 file clean sau lọc similarity
CLEAN_INPUT_DIR = "./outputs/spcqa_similarity_safe_filter_output"

# Thư mục lưu file gộp
MERGED_OUTPUT_DIR = "./outputs/final_data"
os.makedirs(MERGED_OUTPUT_DIR, exist_ok=True)

CLEAN_FILE_NAMES = {
    "ca_chua": "ca_chua_spcqa_v4_similarity_safe_clean.csv",
    "bi_do": "bi_do_spcqa_v4_similarity_safe_clean.csv",
    "kho_qua": "kho_qua_spcqa_v4_similarity_safe_clean.csv",
    "dua_leo": "dua_leo_spcqa_v4_similarity_safe_clean.csv",
}

merged_dfs = []

for crop_key, file_name in CLEAN_FILE_NAMES.items():
    file_path = os.path.join(CLEAN_INPUT_DIR, file_name)

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    df = pd.read_csv(file_path)

    print(f"{crop_key}: {df.shape}")

    # Kiểm tra schema giống nhau
    if crop_key == "ca_chua":
        BASE_COLS_MERGE = df.columns.tolist()
    else:
        assert df.columns.tolist() == BASE_COLS_MERGE, f"{crop_key} bị lệch schema."

    merged_dfs.append(df)

# Gộp 4 cây
merged_df = pd.concat(merged_dfs, ignore_index=True)

# Kiểm tra duplicate cột
assert merged_df.columns.duplicated().sum() == 0, "File gộp bị duplicate cột."

# Kiểm tra duplicate sample_id
duplicate_sample_id = merged_df["sample_id"].duplicated().sum()
print("Duplicate sample_id:", duplicate_sample_id)

if duplicate_sample_id > 0:
    print("Có sample_id bị trùng. Đang tạo sample_id mới theo crop để đảm bảo unique...")
    merged_df["sample_id"] = (
        merged_df["crop"].astype(str).str.upper().str.replace(" ", "_", regex=False)
        + "_MERGED_"
        + merged_df.index.astype(str).str.zfill(6)
    )

# Kiểm tra duplicate nội dung, không tính sample_id
CONTENT_COLS_MERGE = [c for c in merged_df.columns if c != "sample_id"]
duplicate_content = merged_df.duplicated(subset=CONTENT_COLS_MERGE).sum()
print("Duplicate content excluding sample_id:", duplicate_content)

# Lưu file gộp
merged_output_path = os.path.join(
    MERGED_OUTPUT_DIR,
    "all_crops_spcqa_v4_similarity_safe_clean_merged.csv"
)

merged_df.to_csv(merged_output_path, index=False, encoding="utf-8-sig")

print("=" * 80)
print("Đã gộp xong 4 cây.")
print("Tổng số dòng:", len(merged_df))
print("Tổng số cột:", len(merged_df.columns))
print("File đã lưu tại:")
print(merged_output_path)

display(merged_df.head())